# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library. We will review the Croissant metadata, explore available record sets, load records with their specific `@id`s, and perform exploratory analysis and visualization.

### Dataset Source
The dataset source is available as a Croissant schema:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant
!pip install matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show high-level metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Explore available record sets, their field `@id`s, and basic metadata.

In [ ]:
# List all record sets and their IDs
overview = []
for rs in metadata.record_sets:
    entry = {
        '@id': rs['@id'],
        'name': rs.get('name', '(unnamed)'),
        'description': rs.get('description', ''),
        'fields': [(f['@id'], f.get('name', '(unnamed)'), f.get('dataType', '')) for f in rs['fields']]
    }
    overview.append(entry)
# Pretty print the available record sets and their fields
for rs in overview:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs['name']}")
    print(f"  Description: {rs['description']}")
    print("  Fields:")
    for f in rs['fields']:
        print(f"    - Field @id: {f[0]}, name: {f[1]}, dataType: {f[2]}")
    print()

> **Note:** Use the `@id` of the record set and the `@id` of the field(s) for all downstream operations.

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: load all record sets as DataFrames
# Define the record set @ids in the dataset (replace with those from the overview above)
record_sets = [rs['@id'] for rs in overview]
dataframes = {}

for record_set_id in record_sets:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# For demo: print columns for the first record set
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Fields (columns) in DataFrame for record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, and group aggregation.

Select a numeric field and a categorical field from the main record set for demonstration.

In [ ]:
# --- Customize these @id names based on the record set overview ---
record_set_id = main_record_set_id
# For this example, suppose the numeric field is 'gad7_score' and group field is 'gender'
# Use the actual '@id' values as shown in the overview printed above (replace as necessary)
numeric_field_id = None
group_field_id = None
for rs in overview:
    if rs['@id'] == record_set_id:
        for f in rs['fields']:
            if 'gad' in f[0].lower() or 'phq' in f[0].lower() or 'psq' in f[0].lower():
                numeric_field_id = f[0]
            if 'gender' in f[0].lower():
                group_field_id = f[0]
        break
print(f"Selected numeric field `@id`: {numeric_field_id}")
print(f"Selected group field `@id`: {group_field_id}")

# Proceed only if fields are found
if numeric_field_id and (numeric_field_id in dataframes[record_set_id].columns):
    df = dataframes[record_set_id]
    # Basic cleaning: drop missing values for analysis
    df_eda = df.dropna(subset=[numeric_field_id])

    # Filter for values above a nominal threshold (e.g., 10)
    threshold = 10
    filtered_df = df_eda[df_eda[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the numeric field
    filtered_df[numeric_field_id + "_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} (first few records):")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by the categorical field (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        # Aggregate mean normalized score per group
        grouped = (
            filtered_df.groupby(group_field_id)[numeric_field_id + '_normalized'].mean()
        )
        print(f"Grouped mean normalized {numeric_field_id} by {group_field_id}:")
        display(grouped)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions and relationships using the selected numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and group_field_id and numeric_field_id in dataframes[record_set_id].columns:
    plt.figure(figsize=(8, 6))
    sns.histplot(data=dataframes[record_set_id], x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.boxplot(data=dataframes[record_set_id], x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id} in record set {record_set_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("Not enough information for visualization.")

## 6. Conclusion
We have successfully loaded the FAIR² mental health survey dataset, explored its structure and metadata, extracted tabular data using record set and field `@id`s, performed basic analysis, and visualized distributions of key variables. This approach ensures reproducibility and leverages the precise referencing enabled by the [mlcroissant](https://pypi.org/project/mlcroissant/) ecosystem and the Croissant schema standard.

**Key takeaways:**
- The use of Croissant `@id`s ensures unambiguous reference to dataset elements.
- The dataset allows fine-grain exploratory data analysis by demographic and mental health indicators.
- The approach here can be applied to any dataset published in the Croissant format with `mlcroissant`, with minimal changes.

For further data processing, refer to downstream analysis or ML tasks using the loaded DataFrames and metadata.